# Retail Sales & Inventory Exploratory Data Analysis

## 1. Introduction

This notebook contains the exploratory data analysis process for a retail sales and inventory dataset. The purpose of this analysis is to understand sales behavior, product performance, store-level trends, and inventory-related opportunities.

The final goal is to generate business insights that can support decision-making in a retail environment.


## Core Business Questions

1. What are the overall sales trends over time?
2. Which stores generate the highest and lowest revenue?
3. Which products and categories drive most of the sales?
4. Which products have high sales but low inventory?
5. Which products have high inventory but low sales?
6. Which stores show signs of overstocking or understocking?
7. Are there mismatches between inventory availability and sales performance?
8. What business actions can be recommended based on the findings?

1. Load all tables
2. Inspect shapes, columns and data types
3. Check missing values and duplicates
4. Identify relationships between tables
5. Create merged analysis table
6. Calculate revenue
7. Analyze sales by date
8. Analyze sales by store
9. Analyze sales by product/category
10. Analyze inventory by store/product/category
11. Compare sales vs inventory
12. Write insights and recommendations

## Suggested Metrics

- Total Revenue
- Total Units Sold
- Number of Transactions
- Average Units per Transaction
- Revenue by Store
- Revenue by Product
- Revenue by Category
- Units Sold by Product
- Current Stock by Store
- Current Stock by Product
- Inventory-to-Sales Ratio
- Potential Overstock Flag
- Potential Stockout Risk Flag

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

In [4]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Figures directory:", FIGURES_DIR)

Project directory: /Users/farid/Documents/DataAnalyticsPortfolio/retail_sales_inventory_analysis
Raw data directory: /Users/farid/Documents/DataAnalyticsPortfolio/retail_sales_inventory_analysis/data/raw
Figures directory: /Users/farid/Documents/DataAnalyticsPortfolio/retail_sales_inventory_analysis/reports/figures


In [6]:
fct_sales = pd.read_csv(RAW_DATA_DIR / "Maven Toys Data/sales.csv")
dim_data_catalog = pd.read_csv(RAW_DATA_DIR / "Maven Toys Data/data_dictionary.csv")
dim_products = pd.read_csv(RAW_DATA_DIR / "Maven Toys Data/products.csv")
dim_stores = pd.read_csv(RAW_DATA_DIR / "Maven Toys Data/stores.csv")
dim_calendar = pd.read_csv(RAW_DATA_DIR / "Maven Toys Data/calendar.csv")

In [9]:
fct_sales.dtypes

Sale_ID       int64
Date            str
Store_ID      int64
Product_ID    int64
Units         int64
dtype: object

In [10]:
fct_sales.isna().sum()

Sale_ID       0
Date          0
Store_ID      0
Product_ID    0
Units         0
dtype: int64

In [22]:
fct_sales.sample(5)#['Store_ID'].unique()

,Sale_ID,Date,Store_ID,Product_ID,Units
245600,245601,2022-08-12,16,7,1
100253,100254,2022-04-10,35,1,1
827773,827774,2023-09-30,37,7,1
598238,598239,2023-04-30,23,15,1
167151,167152,2022-06-04,33,6,1


In [24]:
dim_products.head()

,Product_ID,Product_Name,Product_Category,Product_Cost,Product_Price
0,1,Action Figure,Toys,$9.99,$15.99
1,2,Animal Figures,Toys,$9.99,$12.99
2,3,Barrel O' Slime,Art & Crafts,$1.99,$3.99
3,4,Chutes & Ladders,Games,$9.99,$12.99
4,5,Classic Dominoes,Games,$7.99,$9.99


In [29]:
sales = fct_sales.merge(dim_products, on="Product_ID", how="left")\
                 .merge(dim_stores, on="Store_ID", how="left")

In [38]:
sales['Date'] = pd.to_datetime(sales['Date'], format='%Y-%m-%d')
sales['Store_Open_Date'] = pd.to_datetime(sales['Store_Open_Date'], format='%Y-%m-%d')
sales['Product_Cost'] = sales['Product_Cost'].apply(lambda x: x.strip('$')).astype(float)
sales['Product_Price'] = sales['Product_Price'].apply(lambda x: x.strip('$')).astype(float)

In [39]:
sales

,Sale_ID,Date,Store_ID,Product_ID,Units,Product_Name,Product_Category,Product_Cost,Product_Price,Store_Name,Store_City,Store_Location,Store_Open_Date
0,1,2022-01-01,24,4,1,Chutes & Ladders,Games,9.99,12.99,Maven Toys Aguascalientes 1,Aguascalientes,Downtown,2010-07-31
1,2,2022-01-01,28,1,1,Action Figure,Toys,9.99,15.99,Maven Toys Puebla 2,Puebla,Downtown,2011-04-01
2,3,2022-01-01,6,8,1,Deck Of Cards,Games,3.99,6.99,Maven Toys Mexicali 1,Mexicali,Commercial,2003-12-13
3,4,2022-01-01,48,7,1,Dart Gun,Sports & Outdoors,11.99,15.99,Maven Toys Saltillo 2,Saltillo,Commercial,2016-03-23
4,5,2022-01-01,44,18,1,Lego Bricks,Toys,34.99,39.99,Maven Toys Puebla 3,Puebla,Residential,2014-12-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...
829257,829258,2023-09-30,24,19,1,Magic Sand,Art & Crafts,13.99,15.99,Maven Toys Aguascalientes 1,Aguascalientes,Downtown,2010-07-31
829258,829259,2023-09-30,16,35,1,Uno Card Game,Games,3.99,7.99,Maven Toys San Luis Potosi 1,San Luis Potosi,Downtown,2007-05-19
829259,829260,2023-09-30,22,19,1,Magic Sand,Art & Crafts,13.99,15.99,Maven Toys Guanajuato 2,Guanajuato,Commercial,2010-03-29
829260,829261,2023-09-30,13,2,2,Animal Figures,Toys,9.99,12.99,Maven Toys Mexicali 2,Mexicali,Downtown,2006-08-30
